In [1]:
import pandas as pd
import numpy as np
import time

from sklearn.ensemble import RandomForestClassifier

In [2]:
caminho_pasta_tratado = '../../dataset tratado/lycos-cicids2017/'

# Cenário Completo
nome_treino = 'Sem Redução de Dimensionalidade/lycos_cicids2017_treinamento.csv'
nome_teste = 'Sem Redução de Dimensionalidade/lycos_cicids2017_teste.csv'

# Sem PortScan no treino
nome_treino_sem_portscan = 'Sem Redução de Dimensionalidade/lycos_cicids2017_treinamento_sem_portscan.csv'
nome_teste_com_portscan = 'Sem Redução de Dimensionalidade/lycos_cicids2017_teste_com_portscan.csv'

# Sem XSS no treino
nome_treino_sem_xss = 'Sem Redução de Dimensionalidade/lycos_cicids2017_treinamento_sem_XSS.csv'
nome_teste_com_xss = 'Sem Redução de Dimensionalidade/lycos_cicids2017_teste_com_XSS.csv'

# Outputs - Completo
nome_treino_reduzidos = 'Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos.csv'
nome_teste_reduzidos = 'Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos.csv'

# Outputs - Sem PortScan
nome_treino_reduzidos_sem_portscan = 'Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos_sem_portscan.csv'
nome_teste_reduzidos_com_portscan = 'Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos_com_portscan.csv'

# Outputs - Sem XSS
nome_treino_reduzidos_sem_xss = 'Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos_sem_xss.csv'
nome_teste_reduzidos_com_xss = 'Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos_com_xss.csv'

# Outputs - Balanced
nome_treino_reduzidos_balanced = 'Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos_balanceados.csv'
nome_teste_reduzidos_balanced = 'Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos_balanceados.csv'

In [3]:
print("Carregando os datasets tratados em CSV...")

df_treino = pd.read_csv(caminho_pasta_tratado + nome_treino)
df_teste = pd.read_csv(caminho_pasta_tratado + nome_teste)

df_treino_sem_portscan = pd.read_csv(caminho_pasta_tratado + nome_treino_sem_portscan)
df_teste_com_portscan = pd.read_csv(caminho_pasta_tratado + nome_teste_com_portscan)

df_treino_sem_xss = pd.read_csv(caminho_pasta_tratado + nome_treino_sem_xss)
df_teste_com_xss = pd.read_csv(caminho_pasta_tratado + nome_teste_com_xss)

print(f"Completo     — Treino: {df_treino.shape} | Teste: {df_teste.shape}")
print(f"Sem PortScan — Treino: {df_treino_sem_portscan.shape} | Teste: {df_teste_com_portscan.shape}")
print(f"Sem XSS      — Treino: {df_treino_sem_xss.shape} | Teste: {df_teste_com_xss.shape}")

Carregando os datasets tratados em CSV...
Completo     — Treino: (1286248, 80) | Teste: (551250, 80)
Sem PortScan — Treino: (1174049, 80) | Teste: (663449, 80)
Sem XSS      — Treino: (1285780, 80) | Teste: (551718, 80)


In [4]:
nomes_enviesados = {
    'Flow ID', 'Source IP', 'Source Port', 'Destination IP',
    'Destination Port', 'Protocol', 'Timestamp', 'timestamp',
    'Fwd Header Length.1'
}

def remover_enviesadas(df_tr, df_te):
    colunas_para_remover = []
    fwd_header_length_vistos = 0
    for coluna in df_tr.columns:
        nome_normalizado = coluna.strip().lower()
        if coluna in nomes_enviesados:
            colunas_para_remover.append(coluna)
        if nome_normalizado == 'fwd header length':
            fwd_header_length_vistos += 1
            if fwd_header_length_vistos > 1:
                colunas_para_remover.append(coluna)
        if nome_normalizado.startswith('fwd header length.'):
            colunas_para_remover.append(coluna)
    colunas_para_remover = [c for c in dict.fromkeys(colunas_para_remover) if c != 'Label']
    df_tr = df_tr.drop(columns=[c for c in colunas_para_remover if c in df_tr.columns])
    df_te = df_te.drop(columns=[c for c in colunas_para_remover if c in df_te.columns])
    return df_tr, df_te

df_treino, df_teste = remover_enviesadas(df_treino, df_teste)
df_treino_sem_portscan, df_teste_com_portscan = remover_enviesadas(df_treino_sem_portscan, df_teste_com_portscan)
df_treino_sem_xss, df_teste_com_xss = remover_enviesadas(df_treino_sem_xss, df_teste_com_xss)

print(f"Completo     — Treino: {df_treino.shape} | Teste: {df_teste.shape}")
print(f"Sem PortScan — Treino: {df_treino_sem_portscan.shape} | Teste: {df_teste_com_portscan.shape}")
print(f"Sem XSS      — Treino: {df_treino_sem_xss.shape} | Teste: {df_teste_com_xss.shape}")

X_treino = df_treino.drop('Label', axis=1)
y_treino = df_treino['Label']
X_teste = df_teste.drop('Label', axis=1)
y_teste = df_teste['Label']

X_treino_sem_portscan = df_treino_sem_portscan.drop('Label', axis=1)
y_treino_sem_portscan = df_treino_sem_portscan['Label']
X_teste_com_portscan = df_teste_com_portscan.drop('Label', axis=1)
y_teste_com_portscan = df_teste_com_portscan['Label']

X_treino_sem_xss = df_treino_sem_xss.drop('Label', axis=1)
y_treino_sem_xss = df_treino_sem_xss['Label']
X_teste_com_xss = df_teste_com_xss.drop('Label', axis=1)
y_teste_com_xss = df_teste_com_xss['Label']

Completo     — Treino: (1286248, 79) | Teste: (551250, 79)
Sem PortScan — Treino: (1174049, 79) | Teste: (663449, 79)
Sem XSS      — Treino: (1285780, 79) | Teste: (551718, 79)


In [5]:
print("--- Cenário Completo: MDI ---")
rf_fs = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
inicio_fs = time.time()
rf_fs.fit(X_treino, y_treino)
fim_fs = time.time()
print(f"Treinamento RF concluído! Tempo: {(fim_fs - inicio_fs):.4f}s")

df_importancias = pd.DataFrame({'Feature': X_treino.columns, 'Importancia': rf_fs.feature_importances_})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)
top_features_completo = df_importancias.head(51)['Feature'].tolist()

print(f"\nTotal: {len(X_treino.columns)} features | Selecionadas: {len(top_features_completo)}")
print("\nTop 10 features mais importantes:")
display(df_importancias.head(10))

--- Cenário Completo: MDI ---
Treinamento RF concluído! Tempo: 39.2053s

Total: 78 features | Selecionadas: 51

Top 10 features mais importantes:


,Feature,Importancia
9,pkt_len_std,0.090683
5,pkt_len_max,0.076280
8,pkt_len_var,0.071227
56,flag_rst,0.070177
25,bwd_pkt_len_max,0.047641
18,fwd_pkt_len_mean,0.043389
16,fwd_pkt_len_max,0.042887
15,fwd_pkt_len_tot,0.040881
27,bwd_pkt_len_mean,0.040162
28,bwd_pkt_len_std,0.035674


In [6]:
df_treino_reduzido = X_treino[top_features_completo].copy()
df_treino_reduzido['Label'] = y_treino.values
df_teste_reduzido = X_teste[top_features_completo].copy()
df_teste_reduzido['Label'] = y_teste.values

print(f"Novas dimensões — Treino: {df_treino_reduzido.shape} | Teste: {df_teste_reduzido.shape}")
df_treino_reduzido.to_csv(caminho_pasta_tratado + nome_treino_reduzidos, index=False)
print(f"Treino salvo: {caminho_pasta_tratado + nome_treino_reduzidos}")
df_teste_reduzido.to_csv(caminho_pasta_tratado + nome_teste_reduzidos, index=False)
print(f"Teste salvo: {caminho_pasta_tratado + nome_teste_reduzidos}")

Novas dimensões — Treino: (1286248, 52) | Teste: (551250, 52)
Treino salvo: ../../dataset tratado/lycos-cicids2017/Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos.csv
Teste salvo: ../../dataset tratado/lycos-cicids2017/Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos.csv


In [7]:
print("--- Cenário Sem PortScan: MDI ---")
rf_fs = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
inicio_fs = time.time()
rf_fs.fit(X_treino_sem_portscan, y_treino_sem_portscan)
fim_fs = time.time()
print(f"Treinamento RF concluído! Tempo: {(fim_fs - inicio_fs):.4f}s")

df_importancias = pd.DataFrame({'Feature': X_treino_sem_portscan.columns, 'Importancia': rf_fs.feature_importances_})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)
top_features_sem_portscan = df_importancias.head(51)['Feature'].tolist()

print(f"\nTotal: {len(X_treino_sem_portscan.columns)} features | Selecionadas: {len(top_features_sem_portscan)}")
print("\nTop 10 features mais importantes:")
display(df_importancias.head(10))

--- Cenário Sem PortScan: MDI ---
Treinamento RF concluído! Tempo: 36.0083s

Total: 78 features | Selecionadas: 51

Top 10 features mais importantes:


,Feature,Importancia
9,pkt_len_std,0.119438
5,pkt_len_max,0.096848
28,bwd_pkt_len_std,0.083916
8,pkt_len_var,0.070434
25,bwd_pkt_len_max,0.063129
27,bwd_pkt_len_mean,0.059426
18,fwd_pkt_len_mean,0.039232
7,pkt_len_mean,0.036180
15,fwd_pkt_len_tot,0.036020
16,fwd_pkt_len_max,0.030427


In [8]:
df_treino_reduzido = X_treino_sem_portscan[top_features_sem_portscan].copy()
df_treino_reduzido['Label'] = y_treino_sem_portscan.values
df_teste_reduzido = X_teste_com_portscan[top_features_sem_portscan].copy()
df_teste_reduzido['Label'] = y_teste_com_portscan.values

print(f"Novas dimensões — Treino: {df_treino_reduzido.shape} | Teste: {df_teste_reduzido.shape}")
df_treino_reduzido.to_csv(caminho_pasta_tratado + nome_treino_reduzidos_sem_portscan, index=False)
print(f"Treino salvo: {caminho_pasta_tratado + nome_treino_reduzidos_sem_portscan}")
df_teste_reduzido.to_csv(caminho_pasta_tratado + nome_teste_reduzidos_com_portscan, index=False)
print(f"Teste salvo: {caminho_pasta_tratado + nome_teste_reduzidos_com_portscan}")

Novas dimensões — Treino: (1174049, 52) | Teste: (663449, 52)
Treino salvo: ../../dataset tratado/lycos-cicids2017/Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos_sem_portscan.csv
Teste salvo: ../../dataset tratado/lycos-cicids2017/Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos_com_portscan.csv


In [9]:
print("--- Cenário Sem XSS: MDI ---")
rf_fs = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
inicio_fs = time.time()
rf_fs.fit(X_treino_sem_xss, y_treino_sem_xss)
fim_fs = time.time()
print(f"Treinamento RF concluído! Tempo: {(fim_fs - inicio_fs):.4f}s")

df_importancias = pd.DataFrame({'Feature': X_treino_sem_xss.columns, 'Importancia': rf_fs.feature_importances_})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)
top_features_sem_xss = df_importancias.head(51)['Feature'].tolist()

print(f"\nTotal: {len(X_treino_sem_xss.columns)} features | Selecionadas: {len(top_features_sem_xss)}")
print("\nTop 10 features mais importantes:")
display(df_importancias.head(10))

--- Cenário Sem XSS: MDI ---
Treinamento RF concluído! Tempo: 37.3867s

Total: 78 features | Selecionadas: 51

Top 10 features mais importantes:


,Feature,Importancia
5,pkt_len_max,0.091266
9,pkt_len_std,0.088810
56,flag_rst,0.074170
8,pkt_len_var,0.058282
27,bwd_pkt_len_mean,0.051468
7,pkt_len_mean,0.049570
72,fwd_subflow_bytes_mean,0.042344
20,fwd_pkt_hdr_len_tot,0.038226
15,fwd_pkt_len_tot,0.037327
16,fwd_pkt_len_max,0.036420


In [10]:
df_treino_reduzido = X_treino_sem_xss[top_features_sem_xss].copy()
df_treino_reduzido['Label'] = y_treino_sem_xss.values
df_teste_reduzido = X_teste_com_xss[top_features_sem_xss].copy()
df_teste_reduzido['Label'] = y_teste_com_xss.values

print(f"Novas dimensões — Treino: {df_treino_reduzido.shape} | Teste: {df_teste_reduzido.shape}")
df_treino_reduzido.to_csv(caminho_pasta_tratado + nome_treino_reduzidos_sem_xss, index=False)
print(f"Treino salvo: {caminho_pasta_tratado + nome_treino_reduzidos_sem_xss}")
df_teste_reduzido.to_csv(caminho_pasta_tratado + nome_teste_reduzidos_com_xss, index=False)
print(f"Teste salvo: {caminho_pasta_tratado + nome_teste_reduzidos_com_xss}")

Novas dimensões — Treino: (1285780, 52) | Teste: (551718, 52)
Treino salvo: ../../dataset tratado/lycos-cicids2017/Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos_sem_xss.csv
Teste salvo: ../../dataset tratado/lycos-cicids2017/Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos_com_xss.csv


In [11]:
print("--- Cenário Balanced (class_weight='balanced'): MDI ---")
rf_fs = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, class_weight='balanced')
inicio_fs = time.time()
rf_fs.fit(X_treino, y_treino)
fim_fs = time.time()
print(f"Treinamento RF concluído! Tempo: {(fim_fs - inicio_fs):.4f}s")

df_importancias = pd.DataFrame({'Feature': X_treino.columns, 'Importancia': rf_fs.feature_importances_})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)
top_features_balanced = df_importancias.head(51)['Feature'].tolist()

print(f"\nTotal: {len(X_treino.columns)} features | Selecionadas: {len(top_features_balanced)}")
print("\nTop 10 features mais importantes:")
display(df_importancias.head(10))

--- Cenário Balanced (class_weight='balanced'): MDI ---
Treinamento RF concluído! Tempo: 32.9019s

Total: 78 features | Selecionadas: 51

Top 10 features mais importantes:


,Feature,Importancia
0,src_port,0.041654
1,dst_port,0.039278
25,bwd_pkt_len_max,0.039001
3,flow_duration,0.034381
27,bwd_pkt_len_mean,0.033061
5,pkt_len_max,0.032568
16,fwd_pkt_len_max,0.030279
4,down_up_ratio,0.027519
20,fwd_pkt_hdr_len_tot,0.027230
19,fwd_pkt_len_std,0.026624


In [12]:
df_treino_reduzido = X_treino[top_features_balanced].copy()
df_treino_reduzido['Label'] = y_treino.values
df_teste_reduzido = X_teste[top_features_balanced].copy()
df_teste_reduzido['Label'] = y_teste.values

print(f"Novas dimensões — Treino: {df_treino_reduzido.shape} | Teste: {df_teste_reduzido.shape}")
df_treino_reduzido.to_csv(caminho_pasta_tratado + nome_treino_reduzidos_balanced, index=False)
print(f"Treino salvo: {caminho_pasta_tratado + nome_treino_reduzidos_balanced}")
df_teste_reduzido.to_csv(caminho_pasta_tratado + nome_teste_reduzidos_balanced, index=False)
print(f"Teste salvo: {caminho_pasta_tratado + nome_teste_reduzidos_balanced}")

Novas dimensões — Treino: (1286248, 52) | Teste: (551250, 52)
Treino salvo: ../../dataset tratado/lycos-cicids2017/Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos_balanceados.csv
Teste salvo: ../../dataset tratado/lycos-cicids2017/Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos_balanceados.csv
